# Reporting Views — Construction Analytics
Creates SQL views over Gold tables: executive summary, cost overruns, schedule risk, and subcontractor scorecard.

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_executive_summary AS
SELECT
    ps.region,
    ps.project_type,
    COUNT(ps.project_id)                                         AS total_projects,
    SUM(CASE WHEN ps.status = 'In Progress' THEN 1 ELSE 0 END)  AS active_projects,
    SUM(CASE WHEN ps.status = 'Completed' THEN 1 ELSE 0 END)    AS completed_projects,
    ROUND(AVG(ps.schedule_variance_days), 1)                     AS avg_schedule_variance_days,
    ROUND(AVG(ps.cost_variance_pct), 2)                          AS avg_cost_variance_pct,
    ROUND(SUM(ps.budget), 2)                                     AS total_budget,
    ROUND(SUM(ps.total_actual_cost), 2)                          AS total_actual_cost,
    ROUND(AVG(ps.avg_pct_complete), 1)                           AS avg_pct_complete,
    SUM(CASE WHEN ps.performance_band = 'Critical' THEN 1 ELSE 0 END) AS critical_projects
FROM gold_portfolio_scorecards ps
GROUP BY ps.region, ps.project_type
""")
print('Created vw_executive_summary')

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_cost_overruns AS
SELECT
    ca.project_id,
    ps.project_name,
    ps.project_type,
    ps.region,
    ps.status,
    ca.cost_category,
    ca.total_planned,
    ca.total_actual,
    ca.total_variance,
    ca.avg_variance_pct,
    ca.overrun_rate,
    ps.cost_risk_band
FROM gold_cost_analysis ca
JOIN gold_portfolio_scorecards ps ON ca.project_id = ps.project_id
WHERE ca.total_variance > 0
ORDER BY ca.total_variance DESC
""")
print('Created vw_cost_overruns')

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_schedule_risk AS
SELECT
    project_id,
    project_name,
    project_type,
    region,
    status,
    planned_duration_days,
    schedule_variance_days,
    schedule_risk_band,
    total_tasks,
    delayed_tasks,
    avg_pct_complete,
    forecast_end_date
FROM gold_portfolio_scorecards
WHERE schedule_risk_band IN ('Critical', 'High', 'Medium')
ORDER BY schedule_variance_days DESC
""")
print('Created vw_schedule_risk')

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_subcontractor_scorecard AS
SELECT
    sp.subcontractor_id,
    sp.company_name,
    sp.trade,
    sp.rating,
    sp.rating_band,
    sp.total_tasks,
    sp.completed_tasks,
    sp.delayed_tasks,
    sp.avg_schedule_variance_days,
    sp.avg_pct_complete,
    sp.on_time_rate
FROM gold_subcontractor_performance sp
ORDER BY sp.on_time_rate DESC
""")
print('Created vw_subcontractor_scorecard')

In [ ]:
print('All reporting views created.')
for v in ['vw_executive_summary', 'vw_cost_overruns', 'vw_schedule_risk', 'vw_subcontractor_scorecard']:
    cnt = spark.sql(f'SELECT COUNT(*) AS n FROM {v}').collect()[0]['n']
    print(f'  {v}: {cnt} rows')